# 11장 파인튜닝

이 장의 예제와 실습을 실행하는 노트북입니다. 개념 설명은 HTML 교재 `11_파인튜닝.html`에 있습니다. 위에서부터 차례로 실행합니다. 실습의 풀이는 교재의 실습 상자에 접혀 있습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(1장의 공개 LLM, 임베딩 모델)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 장의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install transformers torch peft
# 11장의 예제는 1장의 공개 모델 Qwen2.5-0.5B-Instruct를 PC의 CPU에서 학습합니다. 처음 한 번 약 1GB를 내려받습니다
# API 키가 없어도 예제와 실습은 실행할 수 있습니다. 아래 연결 확인에서 오류가 나도
# "학습 데이터" 줄이 출력됐으면 계속 진행합니다. 키가 없으면 "준비 (1)"의 입력 칸에서 Enter만 누릅니다
import csv

import embed
import torch

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
embed.ask_endpoint(MODEL)                         # 모델이 PC에 없고 HF_ENDPOINT가 비어 있으면 주소를 묻습니다
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as transformers_logging
transformers_logging.disable_progress_bar()
torch.manual_seed(0)                              # LoRA 행렬의 초깃값을 실행할 때마다 같게 합니다
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32)   # CPU는 bf16 계산이 느려 fp32로 올립니다
model.generation_config.update(do_sample=False, temperature=None, top_p=None, top_k=None)  # 언제나 1위 토큰을 고릅니다

with open(f'{DATA}/알람.csv', encoding='utf-8') as file:
    alarms = list(csv.DictReader(file))           # 설비 알람 96건. 열: 번호, 알람, 표준, 용도
train_rows = [row for row in alarms if row['용도'] == '학습']
eval_rows = [row for row in alarms if row['용도'] == '평가']
SYSTEM = '설비 알람을 사내 표준 형식으로 바꿔 답한다.'


def chat_prompt(alarm, system=SYSTEM):
    """시스템 프롬프트와 알람을 채팅 템플릿에 넣고, 모델의 답 차례를 여는 표시까지 붙인 글"""
    messages = [{'role': 'system', 'content': system}, {'role': 'user', 'content': alarm}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def make_batch(rows, max_length=128):
    """학습 데이터 여러 건을 패딩한 배치로 만듭니다. 손실은 답(표준 형식) 부분에서만 계산합니다"""
    texts = [chat_prompt(row['알람']) + row['표준'] + '<|im_end|>' for row in rows]
    batch = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    labels = batch['input_ids'].clone()
    for index, row in enumerate(rows):
        prompt_length = len(tokenizer(chat_prompt(row['알람']))['input_ids'])
        labels[index, :prompt_length] = -100      # -100인 위치는 손실 계산에서 뺍니다
    labels[batch['attention_mask'] == 0] = -100   # 패딩 위치도 뺍니다
    batch['labels'] = labels
    return batch


def answer(model, alarm, system=SYSTEM):
    """알람 하나에 대한 모델의 답(최대 20토큰)"""
    model.eval()
    inputs = tokenizer(chat_prompt(alarm, system), return_tensors='pt')
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=20)
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


print(f'PC에서 학습하는 모델: {MODEL}')
print(f'학습 데이터 {len(train_rows)}건, 평가셋 {len(eval_rows)}건. 예: {train_rows[0]["알람"]} → {train_rows[0]["표준"]}')


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 예제 11-1. 패딩과 손실을 계산하는 위치

교재의 절: "학습 데이터가 배치가 되기까지"

학습 데이터 세 건을 배치 하나로 만들고, 건마다 실제 토큰, 패딩, 손실을 계산하는 토큰의 개수와 손실을 계산하는 부분의 글을 출력합니다. 마지막 줄에서는 최대 길이를 40토큰으로 줄였을 때 손실을 계산하는 부분에 무엇이 남는지 봅니다. `make_batch()`는 준비 셀에 있는 함수로, 이 절의 네 단계를 실행합니다.

In [ ]:
batch = make_batch(train_rows[:3])                  # 학습 데이터 3건을 배치 하나로 만듭니다
print('배치의 크기(건수, 토큰 수):', tuple(batch['input_ids'].shape))
for row, ids, mask, labels in zip(train_rows[:3], batch['input_ids'], batch['attention_mask'], batch['labels']):
    real, padding, scored = int(mask.sum()), int((mask == 0).sum()), int((labels != -100).sum())
    print(f"\n{row['알람']}\n  실제 토큰 {real}개, 패딩 {padding}개, 손실을 계산하는 토큰 {scored}개")
    print('  마지막 6토큰:', repr(tokenizer.decode(ids[-6:])))
    print('  손실을 계산하는 부분:', repr(tokenizer.decode(labels[labels != -100])))

short = make_batch(train_rows[:1], max_length=40)   # 최대 길이를 40토큰으로 줄이면
labels = short['labels'][0]
print('\n최대 길이 40토큰일 때 손실을 계산하는 부분:', repr(tokenizer.decode(labels[labels != -100])))

배치의 크기(건수, 토큰 수): (3, 66)

3호기 와이어 본더 본딩 강도 기준 미달 로트 발생
  실제 토큰 66개, 패딩 0개, 손실을 계산하는 토큰 16개
  마지막 6토큰: ' 품질팀<|im_end|>'
  손실을 계산하는 부분: 'WB-03 | 품질 | 품질팀<|im_end|>'

다이싱 소 1호기 스핀들 회전수 알람으로 설비 정지
  실제 토큰 63개, 패딩 3개, 손실을 계산하는 토큰 12개
  마지막 6토큰: '전팀<|im_end|><|endoftext|><|endoftext|><|endoftext|>'
  손실을 계산하는 부분: 'DS-01 | 설비 | 보전팀<|im_end|>'

마킹기 2번 레이저 출력 저하 알람
  실제 토큰 58개, 패딩 8개, 손실을 계산하는 토큰 12개
  마지막 6토큰: '<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>'
  손실을 계산하는 부분: 'MK-02 | 설비 | 보전팀<|im_end|>'

최대 길이 40토큰일 때 손실을 계산하는 부분: ''


## 예제 11-2. LoRA가 학습하는 파라미터 수

교재의 절: "전체 파인튜닝과 LoRA"

0번 층 어텐션의 `q_proj` 행렬 하나에 rank를 바꿔 LoRA를 붙일 때 학습하는 값의 수를 계산하고, 24개 층의 `q_proj`와 `v_proj`에 r=8로 LoRA를 붙인 모델에서 학습하는 파라미터의 비율을 셉니다. 행렬이 클수록 LoRA가 학습하는 비율이 작아지는 것을 봅니다.

In [ ]:
from peft import LoraConfig, get_peft_model

q_proj = model.model.layers[0].self_attn.q_proj     # 0번 층 어텐션의 행렬 하나
d_out, d_in = q_proj.weight.shape
print(f'q_proj 행렬 {d_out}×{d_in}: {q_proj.weight.numel():,}개')
for rank in [1, 8, 64]:
    lora_count = rank * (d_out + d_in)              # B(d_out×r)와 A(r×d_in)의 값의 수
    print(f'  r={rank}: B와 A {lora_count:,}개 ({lora_count / q_proj.weight.numel():.1%})')

# target_modules: LoRA를 붙일 행렬의 이름. 24개 층마다 q_proj와 v_proj에 붙입니다
config = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj', 'v_proj'], task_type='CAUSAL_LM')
lora_model = get_peft_model(model, config)          # 원래 파라미터는 고정하고 LoRA 행렬만 학습 대상으로 둡니다
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in lora_model.parameters())
print(f'\n학습하는 파라미터 {trainable:,}개 / 전체 {total:,}개 = {trainable / total:.2%}')

q_proj 행렬 896×896: 802,816개
  r=1: B와 A 1,792개 (0.2%)
  r=8: B와 A 14,336개 (1.8%)
  r=64: B와 A 114,688개 (14.3%)

학습하는 파라미터 540,672개 / 전체 494,573,440개 = 0.11%


## 예제 11-3. CPU에서 LoRA 학습

교재의 절: "CPU에서 LoRA 학습하기"

예제 11-2에서 LoRA를 붙인 모델을 학습 데이터 64건으로 배치 8건씩, 3에포크(24스텝) 학습합니다. 에포크마다 평균 손실과 걸린 시간을 출력하고, 학습 전후에 평가셋의 알람 세 건에 대한 답을 비교합니다. CPU에서 몇 분 걸립니다.

In [ ]:
import time

optimizer = torch.optim.AdamW([p for p in lora_model.parameters() if p.requires_grad], lr=1e-3)
samples = eval_rows[:3]                                       # 학습에 쓰지 않은 알람
before = [answer(lora_model, row['알람']) for row in samples]   # B가 0이므로 원래 모델의 답과 같습니다
start = time.time()
for epoch in range(3):
    lora_model.train()
    losses = []
    for index in range(0, len(train_rows), 8):                # 배치 8건마다 한 스텝
        loss = lora_model(**make_batch(train_rows[index:index + 8])).loss
        loss.backward()                                       # 기울기를 구하고
        optimizer.step()                                      # 파라미터를 고친 뒤
        optimizer.zero_grad()                                 # 다음 스텝을 위해 기울기를 비웁니다
        losses.append(loss.item())
    print(f'에포크 {epoch + 1}: 평균 손실 {sum(losses) / len(losses):.3f}, 누적 {time.time() - start:.0f}초')
for row, old in zip(samples, before):
    print(f"\n{row['알람']}\n  학습 전: {old[:45]}\n  학습 후: {answer(lora_model, row['알람'])}  (정답 {row['표준']})")

에포크 1: 평균 손실 2.333, 누적 120초
에포크 2: 평균 손실 0.229, 누적 244초
에포크 3: 평균 손실 0.092, 누적 364초

몰딩기 4호기 금형 근처에서 작업자 손 끼임 위험 감지
  학습 전: 물론입니다. 아래는 설비 알람을 사내 표준 형식으로 바꾸
  학습 후: MD-04 | 안전 | 안전팀  (정답 MD-04 | 안전 | 안전팀)

다이 본더 6호기 접착제 보이드 비율 기준 초과
  학습 전: 다이 본더 6호기의 접착제 보이드 비율에 대한 정보
  학습 후: BD-06 | 설비 | 보전팀  (정답 DB-06 | 품질 | 품질팀)

7호기 다이싱 소 냉각수 누수로 바닥 미끄러움
  학습 전: 죄송합니다, 하지만 현재 제게 제공된 정보에 따라 이 문제를 해결하는 방법은
  학습 후: DS-07 | 안전 | 안전팀  (정답 DS-07 | 안전 | 안전팀)


## 실습 11-1. 학습률 바꾸기

교재의 절: "개념 확인 실습"

손실 (w - 3)²에서 w=0으로 시작해 경사 하강법을 다섯 스텝 진행하는 `descend()`를 작성합니다. 학습률 0.01, 0.5, 0.9, 1.1에서 다섯 스텝 뒤의 w가 최솟값 3에 가까워질지, 멀어질지 먼저 `my_guess`에 적습니다. 결과를 보고, 학습률 1.1에서 w가 어떻게 움직였는지 기울기로 설명하면 끝납니다.

In [ ]:
def descend(learning_rate, steps=5, w=0.0):
    """손실 (w - 3)²에 경사 하강법을 steps번 적용하고, 스텝마다의 w를 목록으로 돌려줍니다"""
    history = [w]
    # TODO: 기울기 2 × (w - 3)을 구하고, w에서 학습률 × 기울기를 빼는 일을 steps번 되풀이합니다
    return history


my_guess = {0.01: None, 0.5: None, 0.9: None, 1.1: None}  # TODO: '가까워짐' 또는 '멀어짐'
for learning_rate in [0.01, 0.5, 0.9, 1.1]:
    path = descend(learning_rate)
    print(f"학습률 {learning_rate}: w {' → '.join(f'{w:.2f}' for w in path)}, "
          f'손실 {(path[-1] - 3) ** 2:.2f} (예측 {my_guess[learning_rate]})')

## 실습 11-2. 평가셋과 망각 점검

교재의 절: "개념 확인 실습"

평가셋 32건으로 두 가지 정확도를 측정합니다. 하나는 LoRA를 뗀 원래 모델에 규칙을 모두 적은 시스템 프롬프트(`RULES`)를 준 정확도이고, 다른 하나는 예제 11-3에서 학습한 모델에 짧은 지시만 준 정확도입니다. 이어서 학습과 무관한 질문 하나를 LoRA를 뗀 모델과 학습한 모델에 물어 답을 비교합니다. `accuracy()`를 작성하고 두 정확도를 먼저 `my_guess`에 적습니다. 결과를 보고, 프롬프트와 파인튜닝 중 이 과제에 무엇이 나은지, 학습한 모델이 무엇을 잃었는지 한 문장씩 설명하면 끝납니다. CPU에서 몇 분 걸립니다.

In [ ]:
RULES = ('설비 알람을 사내 표준 형식 "설비코드 | 분류 | 담당"으로 바꿔 답한다.\n'
         '설비코드: 다이싱 소 DS, 다이 본더 DB, 와이어 본더 WB, 몰딩기 MD, 마킹기 MK, 테스트 핸들러 TH 뒤에 '
         '호기 두 자리를 붙인다. 예: 3호기 와이어 본더 → WB-03\n'
         '분류: 사람이 다치거나 다칠 위험이 있으면 안전, 제품의 측정값이나 외관이 기준을 벗어났으면 품질, '
         '그 밖의 설비 동작 이상이면 설비\n'
         '담당: 설비는 보전팀, 품질은 품질팀, 안전은 안전팀\n'
         '예: 3호기 와이어 본더 본딩 강도 기준 미달 로트 발생 → WB-03 | 품질 | 품질팀')


def accuracy(model, rows, system=SYSTEM):
    """rows의 알람마다 answer()로 답을 받아, 표준 열과 글자까지 같은 비율을 돌려줍니다"""
    return 0.0  # TODO


my_guess = {'학습 전, 규칙 프롬프트': None, '학습 후, 짧은 지시': None}  # TODO: 평가셋 정확도를 0~1로 적습니다
with lora_model.disable_adapter():                  # 이 블록 안에서는 LoRA를 뗀 원래 모델이 답합니다
    before = accuracy(lora_model, eval_rows, system=RULES)
after = accuracy(lora_model, eval_rows)
print(f'학습 전, 규칙 프롬프트: {before:.0%} / 학습 후, 짧은 지시: {after:.0%} (예측 {my_guess})')

question = '대한민국의 수도는 어디인가요?'
# TODO: question을 system='질문에 한 문장으로 답한다.'로 LoRA를 뗀 모델과 학습한 모델에 각각 묻고 답을 출력합니다

## 실습 11-3. GPU에서 학습 설정 바꾸기

교재의 절: "개념 확인 실습"

GPU가 있는 환경에서 더 큰 모델(Qwen2.5-1.5B-Instruct)을 LoRA로 학습하고, 에포크마다 학습 데이터와 평가셋의 정확도를 함께 측정해 과적합이 생기는지 봅니다. `EPOCHS`, `LEARNING_RATE`, `RANK`를 바꿔 가며 다시 실행합니다. 학습 데이터의 정확도와 평가셋의 정확도 중 어느 쪽이 더 높을지, 에포크를 늘리면 두 값의 차이가 어떻게 될지 먼저 `my_guess`에 적습니다. 결과를 보고 과적합이 시작된 에포크가 있는지 설명하면 끝납니다. GPU가 없으면 이 셀은 실행하지 않고 넘어갑니다. 실습 11-2의 `accuracy()`를 씁니다.

사내 GPU 환경에서 노트북을 여는 방법: (사내 GPU 환경의 사용 방법을 확인한 뒤 채웁니다.)

In [ ]:
GPU_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'   # 0.5B와 토크나이저가 같아 make_batch(), answer()를 그대로 씁니다
EPOCHS, LEARNING_RATE, RANK = 4, 5e-4, 8   # 바꿔 가며 다시 실행합니다
my_guess = None  # TODO: 두 정확도 중 어느 쪽이 높을지, 에포크를 늘리면 차이가 어떻게 될지 적습니다

if not torch.cuda.is_available():
    print('GPU가 없어 건너뜁니다. GPU가 있는 환경에서 실행합니다')
else:
    gpu_model = AutoModelForCausalLM.from_pretrained(GPU_MODEL, dtype=torch.bfloat16).to('cuda')
    config = LoraConfig(r=RANK, lora_alpha=2 * RANK, target_modules=['q_proj', 'v_proj'], task_type='CAUSAL_LM')
    gpu_lora = get_peft_model(gpu_model, config)
    optimizer = torch.optim.AdamW([p for p in gpu_lora.parameters() if p.requires_grad], lr=LEARNING_RATE)
    for epoch in range(EPOCHS):
        # TODO: 예제 11-3처럼 배치 8건씩 한 에포크를 학습합니다(배치는 .to('cuda')로 GPU에 올립니다)
        # TODO: 학습 데이터 32건과 평가셋 32건의 정확도를 accuracy()로 측정해 출력합니다
        pass